# ML-07 — Baseline Action Score

This notebook builds a simple review rule, checks it on held-out clients, and writes a local queue.


## 1. My rule and reason codes

The baseline puts already-visible pages first. It adds priority when a page has low click-through rate or an average position from 11 to 20. The visibility and low-CTR cutoffs come from the training clients only. More May impressions break ties. This is a review rule, not a claim that a refresh will increase traffic.

The reason codes say whether a page is visible with low CTR, visible near page one, both, or simply visible. A low-visibility page stays lower in the queue until there is more evidence.

## Load the May and June data

I keep the large daily records in DuckDB and bring only page-level results into pandas. The first run saves the results locally. The daily batches keep memory use down.


In [1]:
import getpass

import os

import tempfile

from datetime import date, timedelta

from pathlib import Path

import duckdb

import numpy as np

import pandas as pd

from sklearn.model_selection import GroupShuffleSplit

FEATURE_COLUMNS = (
    'log_prior30_impressions', 'log_prior30_clicks', 'prior30_ctr',
    'prior30_avg_position_filled', 'position_missing',
)

LABEL_END = date(2026, 6, 30)

HISTORICAL_CUTOFF = LABEL_END - timedelta(days=30)

FEATURE_START = HISTORICAL_CUTOFF - timedelta(days=29)

CURRENT_START = HISTORICAL_CUTOFF + timedelta(days=1)

def repo_root() -> Path:
    for path in (Path.cwd(), *Path.cwd().parents):
        if (path / 'work' / 'notebooks').is_dir():
            return path
    raise FileNotFoundError('Open the repository or a folder inside it before running the notebook')

def _quote_path(path: Path) -> str:
    return str(path).replace('\\', '/').replace("'", "''")

def _feature_select(table: str, labeled: bool) -> str:
    extra = (
        ", future30_impressions, CASE WHEN future30_impressions < 0.8 * prior30_impressions "
        "THEN 1 ELSE 0 END AS future_decline_label"
    ) if labeled else ''
    return f"""
        SELECT client_hash_id, content_hash_id, prior30_impressions, prior30_clicks,
               prior30_avg_position,
               LN(1 + prior30_impressions) AS log_prior30_impressions,
               LN(1 + prior30_clicks) AS log_prior30_clicks,
               CAST(prior30_clicks AS DOUBLE) / prior30_impressions AS prior30_ctr,
               COALESCE(prior30_avg_position, 0.0) AS prior30_avg_position_filled,
               CASE WHEN prior30_avg_position IS NULL THEN 1 ELSE 0 END AS position_missing
               {extra}
        FROM {table}
    """

repo = repo_root()


In [2]:
def _summarize_month(con: duckdb.DuckDBPyConnection, source: str,
                     start: date, end: date, folder: Path) -> Path:
    """Stream a month to disk, then deduplicate one day at a time."""
    folder.mkdir()
    stage = folder / 'daily_source.parquet'
    days = folder / 'days'
    days.mkdir()
    summary = folder / 'summary.parquet'
    con.execute(f"""
        COPY (SELECT report_date, client_hash_id, content_hash_id,
                     gsc_impressions, gsc_clicks, gsc_avg_position
              FROM {source}
              WHERE report_date BETWEEN DATE '{start}' AND DATE '{end}')
        TO '{_quote_path(stage)}' (FORMAT PARQUET)
    """)
    day = start
    while day <= end:
        day_path = days / f'{day}.parquet'
        con.execute(f"""
            COPY (
                SELECT client_hash_id, content_hash_id,
                       MIN(gsc_impressions) AS impressions,
                       MIN(gsc_clicks) AS clicks,
                       CASE WHEN MIN(gsc_avg_position) > 0 AND MIN(gsc_impressions) > 0
                            THEN MIN(gsc_avg_position) * MIN(gsc_impressions)
                            ELSE 0 END AS position_numerator,
                       CASE WHEN MIN(gsc_avg_position) > 0 AND MIN(gsc_impressions) > 0
                            THEN MIN(gsc_impressions) ELSE 0 END AS position_denominator,
                       COUNT(*) - 1 AS repeated_rows,
                       CASE WHEN MIN(gsc_impressions) IS DISTINCT FROM MAX(gsc_impressions)
                                      OR MIN(gsc_clicks) IS DISTINCT FROM MAX(gsc_clicks)
                                      OR MIN(gsc_avg_position) IS DISTINCT FROM MAX(gsc_avg_position)
                                      OR (COUNT(gsc_impressions) > 0 AND COUNT(gsc_impressions) < COUNT(*))
                                      OR (COUNT(gsc_clicks) > 0 AND COUNT(gsc_clicks) < COUNT(*))
                                      OR (COUNT(gsc_avg_position) > 0 AND COUNT(gsc_avg_position) < COUNT(*))
                            THEN 1 ELSE 0 END AS conflicting_keys
                FROM read_parquet('{_quote_path(stage)}')
                WHERE report_date = DATE '{day}'
                GROUP BY client_hash_id, content_hash_id
            ) TO '{_quote_path(day_path)}' (FORMAT PARQUET)
        """)
        day += timedelta(days=1)
    stage.unlink()
    con.execute(f"""
        COPY (
            SELECT client_hash_id, content_hash_id, COUNT(*) AS observed_days,
                   COUNT(*) FILTER (WHERE impressions IS NULL OR clicks IS NULL) AS missing_days,
                   SUM(impressions) AS impressions, SUM(clicks) AS clicks,
                   SUM(position_numerator) / NULLIF(SUM(position_denominator), 0) AS avg_position,
                   SUM(repeated_rows) AS repeated_rows,
                   SUM(conflicting_keys) AS conflicting_keys
            FROM read_parquet('{_quote_path(days / '*.parquet')}')
            GROUP BY client_hash_id, content_hash_id
        ) TO '{_quote_path(summary)}' (FORMAT PARQUET)
    """)
    for day_path in days.glob('*.parquet'):
        day_path.unlink()
    days.rmdir()
    return summary


In [3]:
def build_frames(source_override: str | None = None,
                 connection: duckdb.DuckDBPyConnection | None = None
                 ) -> tuple[pd.DataFrame, pd.DataFrame]:
    output = repo_root() / 'work' / 'outputs'
    output.mkdir(parents=True, exist_ok=True)
    con = connection or duckdb.connect()
    with tempfile.TemporaryDirectory(prefix='lane2_build_', dir=output) as scratch:
        temp = Path(scratch)
        spill = output / 'lane2_duckdb_spill'
        spill.mkdir(exist_ok=True)
        con.execute(f"SET temp_directory = '{_quote_path(spill)}'")
        con.execute("SET memory_limit = '1GB'")
        con.execute('SET threads = 2')
        if source_override is None:
            token = os.environ.get('HF_TOKEN') or getpass.getpass('Hugging Face READ token: ')
            con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '"
                        + token.replace("'", "''") + "')")
            del token
            root = 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance'
            may_source = f"read_parquet('{root}/month=2026-05/*.parquet')"
            june_source = f"read_parquet('{root}/month=2026-06/*.parquet')"
        else:
            may_source = june_source = source_override

        may = _summarize_month(con, may_source, FEATURE_START, HISTORICAL_CUTOFF,
                               temp / 'may')
        june = _summarize_month(con, june_source, CURRENT_START, LABEL_END,
                                temp / 'june')
        may_rel = f"read_parquet('{_quote_path(may)}')"
        june_rel = f"read_parquet('{_quote_path(june)}')"
        counts = con.sql(f"""
            SELECT SUM(repeated_rows), SUM(conflicting_keys),
                   COUNT(*) FILTER (WHERE client_hash_id IS NULL OR content_hash_id IS NULL)
            FROM (SELECT * FROM {may_rel} UNION ALL SELECT * FROM {june_rel})
        """).fetchone()
        repeated, conflicting, missing_keys = (int(value or 0) for value in counts)
        assert missing_keys == 0, f'{missing_keys:,} rows have a missing daily key'
        assert conflicting == 0, f'{conflicting:,} daily keys have different GSC values'

        con.execute(f"""
            CREATE OR REPLACE TEMP TABLE historical_metrics AS
            SELECT m.client_hash_id, m.content_hash_id,
                   m.impressions AS prior30_impressions,
                   m.clicks AS prior30_clicks,
                   m.avg_position AS prior30_avg_position,
                   j.impressions AS future30_impressions
            FROM {may_rel} m
            JOIN {june_rel} j USING (client_hash_id, content_hash_id)
            WHERE m.observed_days = 30 AND m.missing_days = 0
              AND m.impressions >= 100
              AND j.observed_days = 30 AND j.missing_days = 0
        """)
        con.execute(f"""
            CREATE OR REPLACE TEMP TABLE current_metrics AS
            SELECT client_hash_id, content_hash_id,
                   impressions AS prior30_impressions,
                   clicks AS prior30_clicks,
                   avg_position AS prior30_avg_position
            FROM {june_rel}
            WHERE observed_days = 30 AND missing_days = 0 AND impressions >= 100
        """)
        historical = con.sql(_feature_select('historical_metrics', labeled=True)).df()
        current = con.sql(_feature_select('current_metrics', labeled=False)).df()

    assert len(historical) > 0 and len(current) > 0, 'No pages met the coverage and impression rules'
    assert historical[list(FEATURE_COLUMNS)].notna().all().all()
    assert current[list(FEATURE_COLUMNS)].notna().all().all()
    print(f'May-June repeated GSC rows removed: {repeated:,}')
    print(f'Historical labeled pages: {len(historical):,}; current June pages: {len(current):,}')
    return historical, current

def load_frames() -> tuple[pd.DataFrame, pd.DataFrame]:
    output = repo_root() / 'work' / 'outputs'
    history_path = output / 'lane2_history_2026-05-31.parquet'
    current_path = output / 'lane2_current_2026-06-30.parquet'
    if history_path.exists() and current_path.exists():
        con = duckdb.connect()
        history = con.sql(f"SELECT * FROM read_parquet('{_quote_path(history_path)}')").df()
        current = con.sql(f"SELECT * FROM read_parquet('{_quote_path(current_path)}')").df()
        print(f'Loaded cached frames: {len(history):,} historical, {len(current):,} current pages')
        return history, current
    history, current = build_frames()
    output.mkdir(parents=True, exist_ok=True)
    con = duckdb.connect()
    con.register('cache_history', history)
    con.register('cache_current', current)
    con.execute(f"COPY cache_history TO '{_quote_path(history_path)}' (FORMAT PARQUET)")
    con.execute(f"COPY cache_current TO '{_quote_path(current_path)}' (FORMAT PARQUET)")
    print('Saved local Parquet caches under work/outputs; dataset files are gitignored')
    return history, current


In [4]:
def grouped_split(frame: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
    groups = frame['client_hash_id']
    if groups.nunique() < 2:
        raise ValueError('A grouped split needs at least two clients')
    train, test = next(GroupShuffleSplit(n_splits=1, test_size=0.25,
                                         random_state=42).split(frame, groups=groups))
    assert set(groups.iloc[train]).isdisjoint(set(groups.iloc[test]))
    return train, test

def baseline_settings(frame: pd.DataFrame, train: np.ndarray) -> tuple[float, float]:
    part = frame.iloc[train]
    return float(part['prior30_impressions'].median()), float(part['prior30_ctr'].quantile(0.25))

def baseline_rank(frame: pd.DataFrame, settings: tuple[float, float]) -> pd.DataFrame:
    visibility_min, low_ctr_max = settings
    visible = frame['prior30_impressions'] >= visibility_min
    low_ctr = frame['prior30_ctr'] <= low_ctr_max
    striking = frame['prior30_avg_position'].between(11, 20, inclusive='both')
    result = frame[['client_hash_id', 'content_hash_id', 'prior30_impressions',
                    'prior30_clicks', 'prior30_ctr', 'prior30_avg_position', 'position_missing']].copy()
    result['baseline_score'] = (visible.astype(int) * (2 + low_ctr.astype(int) + striking.astype(int))
                                + 0.01 * np.log1p(frame['prior30_impressions']))
    result['reason_code'] = np.select(
        [visible & low_ctr & striking, visible & low_ctr, visible & striking, visible],
        ['visible_low_ctr_striking', 'visible_low_ctr', 'visible_striking', 'visible_page'],
        default='low_visibility_review_later',
    )
    return result

def precision_at_k(scores, labels, k: int) -> tuple[float, int]:
    values = np.asarray(scores, dtype=float)
    truth = np.asarray(labels, dtype=int)
    if len(values) != len(truth) or len(values) == 0:
        raise ValueError('Scores and labels need the same nonzero length')
    used = min(k, len(values))
    order = np.argsort(-values, kind='stable')[:used]
    return float(truth[order].mean()), used


In [5]:
from pathlib import Path
import numpy as np

history, current = load_frames()
train_idx, test_idx = grouped_split(history)
settings = baseline_settings(history, train_idx)
history_rank = baseline_rank(history, settings)
print(f'Training-client visibility cutoff: {settings[0]:,.0f} May impressions')
print(f'Training-client low-CTR cutoff: {settings[1]:.2%}')
print(f'Training pages with zero May clicks: '
      f'{(history.prior30_clicks.iloc[train_idx] == 0).mean():.1%}')
print('Reason codes:', ', '.join(sorted(history_rank.reason_code.unique())))

Loaded cached frames: 102,532 historical, 96,678 current pages
Training-client visibility cutoff: 611 May impressions
Training-client low-CTR cutoff: 0.00%
Training pages with zero May clicks: 36.6%
Reason codes: low_visibility_review_later, visible_low_ctr, visible_low_ctr_striking, visible_page, visible_striking


## 2. Ranked queue and baseline check

The historical May queue is checked against the June decline label on clients held out from the cutoff settings. Precision at 20 and 50 is reported beside the June decline base rate. The separate June queue has no future label; it is a list for review at the end of June, not a measured July result. The local CSV is ignored by Git.

In [6]:
heldout = history_rank.iloc[test_idx]
heldout_labels = history.future_decline_label.iloc[test_idx]
print(f'Held-out June decline base rate: {heldout_labels.mean():.1%}')
for k in (20, 50):
    precision, used = precision_at_k(heldout.baseline_score, heldout_labels, k)
    print(f'Baseline precision at {used}: {precision:.1%}')

ranked = baseline_rank(current, settings).sort_values(
    'baseline_score', ascending=False, kind='stable'
).reset_index(drop=True)
output_dir = repo / 'work' / 'outputs'
output_dir.mkdir(parents=True, exist_ok=True)
queue_path = output_dir / 'baseline_action_score.csv'
ranked.to_csv(queue_path, index=False)
print(f'June review queue: {len(ranked):,} pages; saved at {queue_path}')

Held-out June decline base rate: 65.5%
Baseline precision at 20: 95.0%
Baseline precision at 50: 92.0%
June review queue: 96,678 pages; saved at D:\flyrank-ml-internship-starter\work\outputs\baseline_action_score.csv


## 3. Top-20 review

The top 20 lines give an action, reason code, evidence note, and a reason the pick could be wrong. A zero-click cutoff or fewer than 10 clicks makes CTR a weak signal even when the page has many impressions. An editor still needs to check intent, accuracy, seasonality, and whether a change is appropriate. No client or page IDs are printed here.


In [7]:
def review_action(reason):
    if 'low_ctr' in reason:
        return 'Review title and search snippet'
    if 'striking' in reason:
        return 'Review relevance and internal links'
    if reason == 'visible_page':
        return 'Check whether the page is current and complete'
    return 'Collect more evidence before editing'

for rank, row in enumerate(ranked.head(20).itertuples(index=False), start=1):
    if row.position_missing:
        confidence = 'position missing'
        possible_error = 'Average position is unavailable'
    elif row.prior30_clicks < 10:
        confidence = 'few observed clicks'
        possible_error = 'CTR may be noisy with fewer than 10 clicks'
    else:
        confidence = 'measured June search signal'
        possible_error = 'Intent, seasonality, or a recent change may explain the metric' 
    print(f'{rank:2d}. Action: {review_action(row.reason_code)}. '
          f'Reason: {row.reason_code}. Confidence: {confidence}. '
          f'Could be wrong: {possible_error}.')
print('Human page review is still required before any action')

 1. Action: Review title and search snippet. Reason: visible_low_ctr_striking. Confidence: few observed clicks. Could be wrong: CTR may be noisy with fewer than 10 clicks.
 2. Action: Review title and search snippet. Reason: visible_low_ctr_striking. Confidence: few observed clicks. Could be wrong: CTR may be noisy with fewer than 10 clicks.
 3. Action: Review title and search snippet. Reason: visible_low_ctr_striking. Confidence: few observed clicks. Could be wrong: CTR may be noisy with fewer than 10 clicks.
 4. Action: Review title and search snippet. Reason: visible_low_ctr_striking. Confidence: few observed clicks. Could be wrong: CTR may be noisy with fewer than 10 clicks.
 5. Action: Review title and search snippet. Reason: visible_low_ctr_striking. Confidence: few observed clicks. Could be wrong: CTR may be noisy with fewer than 10 clicks.
 6. Action: Review title and search snippet. Reason: visible_low_ctr_striking. Confidence: few observed clicks. Could be wrong: CTR may be n

## 4. Weak picks and leakage check

A high rank can still be weak when position is missing or clicks are sparse. The check counts those cases in the top 20. The score uses the June feature window only for the current queue. June future outcomes, the decline label, IDs, GA4 values, query metrics, and product flags do not enter the rule. The historical precision is a retrospective check, not proof of a refresh effect.

In [8]:
top20 = ranked.head(20)
print(f'Top-20 picks with missing average position: {int(top20.position_missing.sum())}')
print(f'Top-20 picks with fewer than 10 prior clicks: '
      f'{int((top20.prior30_clicks < 10).sum())}')
forbidden = ('future', 'label', 'ga4', 'query', 'url', 'flag')
assert not any(word in column for column in ranked.columns for word in forbidden)
assert 'client_hash_id' not in FEATURE_COLUMNS
assert 'content_hash_id' not in FEATURE_COLUMNS
print('The ranked score contains no future outcome, product flag, or private query field')

Top-20 picks with missing average position: 0
Top-20 picks with fewer than 10 prior clicks: 20
The ranked score contains no future outcome, product flag, or private query field


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under work/notebooks/ — then submit your repo URL on the card. Done.